## 07b - Agent Demo Minimal

### Objetivo
Demo 
- usando una base de conocimiento pequeña y grounded en datos,
- usando prompts hiper-personalizados por escenario,
- con dos modos:
  - **mock local** (costo casi cero),
  - **live API** (demo real con gasto mínimo).

### Escenarios
1. Cliente joven, digital y frecuente.
2. Cliente mayor, conservador o inactivo, con mala experiencia previa.
3. Cliente VIP o de alto gasto.

In [1]:
from pathlib import Path
import sys
import json
import os

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import load_all_datasets
from src.analysis.knowledge_base import (
    build_item_level_sales_base,
    compute_product_sales,
    select_reference_products,
    build_knowledge_base_dict,
    build_prompt_ready_knowledge_base,
)
from src.prompts.system_prompts import (
    SYSTEM_PROMPTS,
    build_purchase_history_summary,
    render_system_prompt,
)
from src.demo.agent_demo import (
    local_mock_response,
    live_openai_response,
)

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"

customer_360_features = pd.read_csv(processed_dir / "customer_360_features.csv")
bundle = load_all_datasets()

In [4]:
## Construcción base para demo
item_base = build_item_level_sales_base(bundle)
product_sales = compute_product_sales(item_base)
reference_products = select_reference_products(product_sales, top_k=5, n_products=3)
knowledge_base = build_knowledge_base_dict(reference_products)
knowledge_base_prompt = build_prompt_ready_knowledge_base(knowledge_base)

reference_products

,product_id,product_category,total_volume,total_revenue,avg_price,n_orders,rank_volume,rank_revenue,selection_score
0,bb50f2e236e5eea0100680137654686c,health_beauty,194,63560.0,327.628866,186,7.0,1.0,1.142857
1,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,37104.3,71.354423,425,1.0,7.0,1.142857
2,6cdd53843498f92890544667809f1595,health_beauty,153,53652.3,350.668627,148,7.0,2.0,0.642857


In [5]:
print(knowledge_base_prompt)

- bb50f2e236e5eea0100680137654686c | categoría: health_beauty | precio promedio: $327.63 | mensaje comercial: Bienestar con alta tracción comercial. Referencia muy atractiva dentro del portafolio de cuidado personal, ideal para campañas de recompra, rutina y autocuidado. En el dataset mostró una tracción destacada con 194 ítems vendidos y 186 órdenes asociadas. Su precio real promedio fue de $327.63, lo que lo convierte en una referencia atractiva para campañas de recomendación y personalización.
- aca2eb7d00ea1a7b8ebd4e68314663af | categoría: furniture_decor | precio promedio: $71.35 | mensaje comercial: Estética y renovación de espacios. Opción con potencial comercial para clientes interesados en transformar sus espacios con soluciones funcionales y visualmente atractivas. En el dataset mostró una tracción destacada con 520 ítems vendidos y 425 órdenes asociadas. Su precio real promedio fue de $71.35, lo que lo convierte en una referencia atractiva para campañas de recomendación y pe

In [6]:
## Selección consistente de escenarios

scenario_1 = (
    customer_360_features
    .query("customer_segment_primary == 'CORE_LOYAL'")
    .sort_values(["frequency", "monetary"], ascending=[False, False])
    .iloc[0]
)

scenario_2_candidates = customer_360_features.loc[
    customer_360_features["customer_flags_text"].fillna("").str.contains("FRICTION_RISK", regex=False)
].copy()

scenario_2 = (
    scenario_2_candidates
    .query("customer_segment_primary in ['CHURN_RISK', 'HIGH_VALUE_RISK']")
    .sort_values(["recency_days", "inactivity_risk_score"], ascending=[False, False])
    .iloc[0]
)

scenario_3 = (
    customer_360_features
    .query("customer_segment_primary == 'VIP'")
    .sort_values(["monetary", "avg_order_value"], ascending=[False, False])
    .iloc[0]
)

In [7]:
scenario_context = pd.DataFrame([
    {
        "scenario_name": "young_digital_frequent",
        "edad": 24,
        "genero": "No especificado",
        "perfil_de_cliente": scenario_1["customer_segment_primary"],
        "flags": scenario_1["customer_flags_text"],
        "historial_compras": build_purchase_history_summary(scenario_1.to_dict()),
        "customer_unique_id": scenario_1["customer_unique_id"],
        "rationale": "Cliente recurrente con comportamiento compatible con un perfil digital y activo.",
    },
    {
        "scenario_name": "older_conservative_friction",
        "edad": 63,
        "genero": "No especificado",
        "perfil_de_cliente": scenario_2["customer_segment_primary"],
        "flags": scenario_2["customer_flags_text"],
        "historial_compras": build_purchase_history_summary(scenario_2.to_dict()),
        "customer_unique_id": scenario_2["customer_unique_id"],
        "rationale": "Cliente de alto riesgo o abandono con señales de fricción en experiencia.",
    },
    {
        "scenario_name": "vip_high_spend",
        "edad": 42,
        "genero": "No especificado",
        "perfil_de_cliente": scenario_3["customer_segment_primary"],
        "flags": scenario_3["customer_flags_text"],
        "historial_compras": build_purchase_history_summary(scenario_3.to_dict()),
        "customer_unique_id": scenario_3["customer_unique_id"],
        "rationale": "Cliente premium de alto valor económico.",
    },
])

scenario_context

,scenario_name,edad,genero,perfil_de_cliente,flags,historial_compras,customer_unique_id,rationale
0,young_digital_frequent,24,No especificado,CORE_LOYAL,"HIGH_VALUE, FRICTION_RISK","Cliente con compra recurrente, última activida...",f0e310a6839dce9de1638e0fe5ab282a,Cliente recurrente con comportamiento compatib...
1,older_conservative_friction,63,No especificado,HIGH_VALUE_RISK,"HIGH_VALUE, FRICTION_RISK, CHURN_SIGNAL","Cliente con compra esporádica, última activida...",830d5b7aaa3b6f1e9ad63703bec97d23,Cliente de alto riesgo o abandono con señales ...
2,vip_high_spend,42,No especificado,VIP,"HIGH_VALUE, FRICTION_RISK","Cliente con compra recurrente, última activida...",c8460e4251689ba205045f3ea17884a1,Cliente premium de alto valor económico.


In [8]:
# render por escenario
selected_idx = 0
selected = scenario_context.loc[selected_idx].to_dict()

prompt_text = render_system_prompt(
    SYSTEM_PROMPTS[selected["scenario_name"]],
    edad=selected["edad"],
    genero=selected["genero"],
    perfil_de_cliente=selected["perfil_de_cliente"],
    historial_compras=selected["historial_compras"],
    base_conocimiento=knowledge_base_prompt,
)

print(prompt_text)

Eres el system prompt de un agente de ecommerce para clientes jóvenes, digitales y frecuentes.

Tu misión es recomendar productos de forma hiper-personalizada, con tono ágil, cercano, actual y orientado a conveniencia.
Habla en español claro, natural y breve. Evita sonar robótico. No uses lenguaje demasiado técnico.

Contexto dinámico del cliente:
- Edad: 24
- Género: No especificado
- Perfil de cliente: CORE_LOYAL
- Historial de compras resumido: Cliente con compra recurrente, última actividad hace 147 días, afinidad principal por la categoría bed_bath_table, gasto histórico aproximado de $438.09, ticket promedio de $73.02 y flags actuales: HIGH_VALUE, FRICTION_RISK.
- Base de conocimiento disponible: - bb50f2e236e5eea0100680137654686c | categoría: health_beauty | precio promedio: $327.63 | mensaje comercial: Bienestar con alta tracción comercial. Referencia muy atractiva dentro del portafolio de cuidado personal, ideal para campañas de recompra, rutina y autocuidado. En el dataset mo

## 1. Demo mock local

In [9]:
profile_payload = {
    "edad": selected["edad"],
    "genero": selected["genero"],
    "perfil_de_cliente": selected["perfil_de_cliente"],
    "historial_compras": selected["historial_compras"],
    "flags": selected["flags"],
}

mock_response = local_mock_response(
    scenario_name=selected["scenario_name"],
    profile=profile_payload,
    knowledge_base=knowledge_base,
)

print(mock_response)

Te comparto opciones muy alineadas con tu estilo de compra reciente.
- producto: bb50f2e236e5eea0100680137654686c | categoría: health_beauty | razón: encaja con tu historial y destaca por su precio promedio de $327.63 y su buen desempeño comercial.
- producto: aca2eb7d00ea1a7b8ebd4e68314663af | categoría: furniture_decor | razón: encaja con tu historial y destaca por su precio promedio de $71.35 y su buen desempeño comercial.
- producto: 6cdd53843498f92890544667809f1595 | categoría: health_beauty | razón: encaja con tu historial y destaca por su precio promedio de $350.67 y su buen desempeño comercial.
Si alguna opción te interesa, puedo ayudarte a priorizar la más conveniente.
